In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('diabetes.csv')
df.head()

In [ ]:
print(f"Dataset shape: {df.shape}")
print(f"\nDataset Info:")
print(df.info())
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nBasic Statistics:\n{df.describe()}")

In [ ]:
print(f"Class Distribution:\n{df.iloc[:, -1].value_counts()}")
print(f"\nClass Distribution (%):\n{df.iloc[:, -1].value_counts(normalize=True) * 100}")

In [ ]:
# DATA PREPROCESSING
numeric_columns = df.select_dtypes(include=[np.number]).columns

# Handle missing values (represented as 0 in diabetes dataset)
zero_columns = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']

for col in zero_columns:
    if col in df.columns:
        df[col] = df[col].replace(0, df[col].replace(0, np.nan).mean())

print("Missing values handled by replacing zeros with column mean")

In [ ]:
# Separate features and target
X = df.iloc[:, :-1]
y = df.iloc[:, -1]

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Training set size: {X_train.shape[0]}")
print(f"Testing set size: {X_test.shape[0]}")

In [ ]:
# NORMALIZE FEATURES
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Features normalized using StandardScaler")

In [ ]:
# SVM MODEL
svm_model = SVC(kernel='rbf', random_state=42, probability=True)
svm_model.fit(X_train_scaled, y_train)

svm_pred_train = svm_model.predict(X_train_scaled)
svm_pred_test = svm_model.predict(X_test_scaled)

print("SVM Model trained successfully")

In [ ]:
# DECISION TREE MODEL
dt_model = DecisionTreeClassifier(max_depth=10, random_state=42, min_samples_split=5)
dt_model.fit(X_train, y_train)

dt_pred_train = dt_model.predict(X_train)
dt_pred_test = dt_model.predict(X_test)

print("Decision Tree Model trained successfully")

In [ ]:
# EVALUATION METRICS
def evaluate_model(y_true_train, y_pred_train, y_true_test, y_pred_test, model_name):
    print(f"\n{'='*50}")
    print(f"{model_name} - EVALUATION RESULTS")
    print(f"{'='*50}")
    
    train_accuracy = accuracy_score(y_true_train, y_pred_train)
    test_accuracy = accuracy_score(y_true_test, y_pred_test)
    
    print(f"\nTraining Accuracy: {train_accuracy:.4f}")
    print(f"Testing Accuracy: {test_accuracy:.4f}")
    print(f"Overfitting Gap: {(train_accuracy - test_accuracy):.4f}")
    
    print(f"\nPrecision (Test): {precision_score(y_true_test, y_pred_test):.4f}")
    print(f"Recall (Test): {recall_score(y_true_test, y_pred_test):.4f}")
    print(f"F1-Score (Test): {f1_score(y_true_test, y_pred_test):.4f}")
    
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_true_test, y_pred_test)}")
    print(f"\nClassification Report:\n{classification_report(y_true_test, y_pred_test)}")
    
    return {
        'model': model_name,
        'train_acc': train_accuracy,
        'test_acc': test_accuracy,
        'precision': precision_score(y_true_test, y_pred_test),
        'recall': recall_score(y_true_test, y_pred_test),
        'f1': f1_score(y_true_test, y_pred_test)
    }

In [ ]:
svm_results = evaluate_model(y_train, svm_pred_train, y_test, svm_pred_test, "SVM Model")

In [ ]:
dt_results = evaluate_model(y_train, dt_pred_train, y_test, dt_pred_test, "Decision Tree Model")

In [ ]:
# CONFUSION MATRICES VISUALIZATION
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

svm_cm = confusion_matrix(y_test, svm_pred_test)
dt_cm = confusion_matrix(y_test, dt_pred_test)

sns.heatmap(svm_cm, annot=True, fmt='d', cmap='Blues', ax=axes[0], cbar=False)
axes[0].set_title('SVM - Confusion Matrix', fontsize=12, fontweight='bold')
axes[0].set_ylabel('True Label')
axes[0].set_xlabel('Predicted Label')

sns.heatmap(dt_cm, annot=True, fmt='d', cmap='Greens', ax=axes[1], cbar=False)
axes[1].set_title('Decision Tree - Confusion Matrix', fontsize=12, fontweight='bold')
axes[1].set_ylabel('True Label')
axes[1].set_xlabel('Predicted Label')

plt.tight_layout()
plt.show()

In [ ]:
# MODEL COMPARISON
metrics_df = pd.DataFrame([svm_results, dt_results])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy Comparison
models = metrics_df['model']
x = np.arange(len(models))
width = 0.35

axes[0].bar(x - width/2, metrics_df['train_acc'], width, label='Training Accuracy', alpha=0.8)
axes[0].bar(x + width/2, metrics_df['test_acc'], width, label='Testing Accuracy', alpha=0.8)
axes[0].set_ylabel('Accuracy', fontsize=11, fontweight='bold')
axes[0].set_title('Training vs Testing Accuracy', fontsize=12, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models)
axes[0].legend()
axes[0].set_ylim([0, 1])

# Metrics Comparison
metrics_names = ['precision', 'recall', 'f1']
x_metrics = np.arange(len(metrics_names))

for i, model in enumerate(models):
    values = [metrics_df.loc[i, metric] for metric in metrics_names]
    axes[1].plot(x_metrics, values, marker='o', label=model, linewidth=2, markersize=8)

axes[1].set_ylabel('Score', fontsize=11, fontweight='bold')
axes[1].set_title('Precision, Recall, and F1-Score Comparison', fontsize=12, fontweight='bold')
axes[1].set_xticks(x_metrics)
axes[1].set_xticklabels(metrics_names)
axes[1].legend()
axes[1].set_ylim([0, 1])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# FEATURE IMPORTANCE (Decision Tree)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': dt_model.feature_importances_
}).sort_values('Importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='steelblue')
plt.xlabel('Importance', fontsize=11, fontweight='bold')
plt.title('Decision Tree - Feature Importance', fontsize=12, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("\nFeature Importance:")
print(feature_importance)

In [ ]:
# ANALYSIS AND DISCUSSION
print("\n" + "="*70)
print("ANALYSIS AND DISCUSSION: OVERFITTING AND ACCURACY TRADE-OFFS")
print("="*70)

print(f"\n1. SVM Model:")
print(f"   - Training Accuracy: {svm_results['train_acc']:.4f}")
print(f"   - Testing Accuracy: {svm_results['test_acc']:.4f}")
print(f"   - Overfitting Gap: {(svm_results['train_acc'] - svm_results['test_acc']):.4f}")
print(f"   - Observations: SVM shows {'minimal' if (svm_results['train_acc'] - svm_results['test_acc']) < 0.1 else 'moderate'} overfitting.")

print(f"\n2. Decision Tree Model:")
print(f"   - Training Accuracy: {dt_results['train_acc']:.4f}")
print(f"   - Testing Accuracy: {dt_results['test_acc']:.4f}")
print(f"   - Overfitting Gap: {(dt_results['train_acc'] - dt_results['test_acc']):.4f}")
print(f"   - Observations: Decision Tree shows {'minimal' if (dt_results['train_acc'] - dt_results['test_acc']) < 0.1 else 'significant'} overfitting.")

best_model = 'SVM' if svm_results['test_acc'] > dt_results['test_acc'] else 'Decision Tree'
print(f"\n3. Best Performing Model: {best_model}")
print(f"   - Better generalization to unseen data")
print(f"   - More reliable for medical predictions")

print(f"\n4. Trade-offs:")
print(f"   - SVM: Good generalization but slower training on large datasets")
print(f"   - Decision Tree: Faster training but prone to overfitting without proper constraints")
print(f"   - For healthcare: Precision and Recall are critical to minimize false positives/negatives")